In [6]:
!ls /kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage
#!ls /kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train

train  val


In [1]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

train_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train"
test_dir = "/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val"

BATCH_SIZE = 64
IMG_SIZE = (224,224)

train_ds = image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)


test_ds = image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
# Uruchom to w nowej komórce na Kaggle (tam gdzie masz dostęp do train_ds)

print("Oto jedyna, matematycznie prawdziwa lista klas Twojego datasetu:\n")
print("class_names = [")
for name in train_ds.class_names:
    print(f'    "{name}",')
print("]")

2026-06-06 18:45:54.493636: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780771554.696748      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780771554.750630      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780771555.190944      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780771555.190986      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780771555.190989      57 computation_placer.cc:177] computation placer alr

Found 43444 files belonging to 38 classes.


I0000 00:00:1780771601.312535      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 10861 files belonging to 38 classes.
Oto jedyna, matematycznie prawdziwa lista klas Twojego datasetu:

class_names = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Lea

In [8]:
from tensorflow.keras import layers, models

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    layers.RandomFlip("horizontal"),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    
    base_model,
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(38, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_1 (RandomFlip)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness_1             │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_1               │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,854 (9.26 MB)

 Trainable params: 168,870 (659.65 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
print("--starting first stage--")

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

--starting first stage--
Epoch 1/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 76s 98ms/step - accuracy: 0.7341 - loss: 1.0044 - val_accuracy: 0.9467 - val_loss: 0.1789
Epoch 2/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 54s 80ms/step - accuracy: 0.9226 - loss: 0.2418 - val_accuracy: 0.9633 - val_loss: 0.1208
Epoch 3/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 53s 78ms/step - accuracy: 0.9413 - loss: 0.1769 - val_accuracy: 0.9703 - val_loss: 0.0959
Epoch 4/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 53s 78ms/step - accuracy: 0.9517 - loss: 0.1481 - val_accuracy: 0.9739 - val_loss: 0.0840
Epoch 5/5
679/679 ━━━━━━━━━━━━━━━━━━━━ 52s 77ms/step - accuracy: 0.9582 - loss: 0.1262 - val_accuracy: 0.9792 - val_loss: 0.0684


In [10]:
print("--preparing model for second stage--")

unfreezeModel = model.layers[4]
unfreezeModel.trainable = True

for layer in unfreezeModel.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

--preparing model for second stage--


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_1 (RandomFlip)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness_1             │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_1               │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,854 (9.26 MB)

 Trainable params: 1,695,270 (6.47 MB)

 Non-trainable params: 731,584 (2.79 MB)

In [11]:
print("--starting second stage - fine-tuning--")

history_f = model.fit(
    train_ds,
    validation_data = train_ds,
    epochs=7
)

--starting second stage - fine-tuning--
Epoch 1/7


2026-05-28 20:08:54.549370: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-28 20:08:54.748057: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


678/679 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.8372 - loss: 0.5474

2026-05-28 20:09:31.639913: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-28 20:09:31.840292: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


679/679 ━━━━━━━━━━━━━━━━━━━━ 88s 104ms/step - accuracy: 0.8373 - loss: 0.5468 - val_accuracy: 0.9684 - val_loss: 0.0940
Epoch 2/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 54s 80ms/step - accuracy: 0.9431 - loss: 0.1747 - val_accuracy: 0.9762 - val_loss: 0.0704
Epoch 3/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 54s 80ms/step - accuracy: 0.9590 - loss: 0.1237 - val_accuracy: 0.9825 - val_loss: 0.0528
Epoch 4/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 55s 80ms/step - accuracy: 0.9659 - loss: 0.1014 - val_accuracy: 0.9862 - val_loss: 0.0427
Epoch 5/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 55s 80ms/step - accuracy: 0.9724 - loss: 0.0790 - val_accuracy: 0.9903 - val_loss: 0.0328
Epoch 6/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 55s 81ms/step - accuracy: 0.9781 - loss: 0.0653 - val_accuracy: 0.9925 - val_loss: 0.0266
Epoch 7/7
679/679 ━━━━━━━━━━━━━━━━━━━━ 55s 81ms/step - accuracy: 0.9816 - loss: 0.0550 - val_accuracy: 0.9944 - val_loss: 0.0214


In [12]:
# Zapisanie wyszkolonego modelu 
model_save_path = '/kaggle/working/plant_disease_detector_v1.keras'
model.save(model_save_path)

print(f"Model został pomyślnie zapisany w: {model_save_path}")

Model został pomyślnie zapisany w: /kaggle/working/plant_disease_detector_v1.keras
